In [15]:
import json
import sys
from tqdm import tqdm
from datetime import datetime
from dataclasses import asdict
# モジュールをリロード
import importlib
import llm_preference_extraction.evaluation.dialogue_evaluator as dialogue_evaluator
importlib.reload(dialogue_evaluator)
from llm_preference_extraction.evaluation.dialogue_evaluator import collect_comparisons, compute_scores
sys.path.insert(0, '..')

## Input

In [2]:
# データセットを読み込む
dataset_path = "../data/ground_truth/test.json"

with open(dataset_path, "r",encoding="utf-8") as f:
    dataset = json.load(f)

print(json.dumps(dataset, indent=2))

[
  {
    "dialogue_id": 0,
    "original_index": 1815,
    "original_dialogue": "System: I think modern painting means nothing .\nUser: I think so too . It's just pointless .\nSystem: Then why are so many crazy about it\nUser: I don't know.Maybe they are really crazy .\nSystem: Maybe .",
    "translated_dialogue": "System: \u73fe\u4ee3\u7d75\u753b\u306a\u3093\u3066\u610f\u5473\u306a\u3044\u3068\u601d\u3046\u3088\u3002\nUser: \u305d\u3046\u601d\u3046\u3088\u3002\u305f\u3060\u305f\u3060\u7121\u610f\u5473\u3060\u3002\nSystem: \u3058\u3083\u3042\u306a\u3093\u3067\u3042\u3093\u306a\u306b\u71b1\u72c2\u3059\u308b\u3093\u3060\uff1f\nUser: \u308f\u304b\u3089\u306a\u3044\u3002\u305f\u3076\u3093\u672c\u5f53\u306b\u982d\u304c\u304a\u304b\u3057\u3044\u3093\u3060\u308d\u3046\u3002\nSystem: \u305d\u3046\u304b\u3082\u306d\u3002",
    "annotations": [
      {
        "entity": "modern painting",
        "axis": "liking",
        "sub_axis": "identification",
        "polarity": "positive",
        "in

## Process

In [3]:
from llm_preference_extraction import extractors as ex

few_shot_id = [0, 1, 2]
model_name = "llama3.1:8b"
base_url = "http://localhost:11434/v1"
api_key = "llama3.1"

# クライアント作成
client = ex.create_client(base_url, api_key)

# プロンプトテンプレート読み込み
few_shot_example_text = ex.create_few_shot_examples(dataset, few_shot_ids=few_shot_id)
prompt_template = ex.load_prompt_template()
system_prompt = prompt_template.replace("{few_shot_example}", few_shot_example_text)

# スキーマ読み込み
schema = ex.load_schema()

# テスト対象の対話を取得(few-shot例を除く)
test_dialogues = [d for d in dataset if d["dialogue_id"] not in few_shot_id]

results = []

for dialogue_data in tqdm(test_dialogues, desc=f"Processing [{model_name}]"):
    dialogue_id = dialogue_data["dialogue_id"]
    dialogue_text = dialogue_data["original_dialogue"]

    # 抽出実行
    extraction_result = ex.extract_preferences(
        client,
        model_name,
        dialogue_text,
        dialogue_id,
        system_prompt,
        schema
    )

    # 元のアノテーションも保持
    result_with_annotation = {
        "dialogue_id": dialogue_id,
        "original_dialogue": dialogue_text,
        "translated_dialogue": dialogue_data.get("translated_dialogue", ""),
        "ground_truth_annotation": dialogue_data.get("annotations", []),
        "extracted_preferences": extraction_result,
    }

    results.append(result_with_annotation)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

experiment_metadata = {
    "experiment_info" : {
        "timestamp": timestamp,
        "dataset_path": dataset_path,
        "few_shot_ids": few_shot_id,
        "total_test_dialogues": len(test_dialogues),
        "model": model_name,
    },
    "results": results,
}

total_extracted = sum(
    len(r["extracted_preferences"].get("preferences", [])) for r in results
)
total_ground_truth = sum(len(r["ground_truth_annotation"]) for r in results)

print(f"\n統計:")
print(f"    - 抽出された嗜好数: {total_extracted}")
print(f"    - Ground truth嗜好数: {total_ground_truth}")

Processing [llama3.1:8b]: 100%|██████████| 8/8 [00:38<00:00,  4.87s/it]


統計:
    - 抽出された嗜好数: 19
    - Ground truth嗜好数: 15


In [4]:
print(json.dumps(results, indent=2))

[
  {
    "dialogue_id": 3,
    "original_dialogue": "System: Can I take your order please ?\nUser: Can I get a burger and a large fries ?\nSystem: Sure . Anything to drink with that ?\nUser: A large coke , please .\nSystem: Eating here or to go ?\nUser: Eating here , please .\nSystem: That's 7 dollars and 25 cents . You can get free refills with your coke .",
    "translated_dialogue": "System: \u3054\u6ce8\u6587\u3092\u304a\u4f3a\u3044\u3057\u3066\u3082\u3088\u308d\u3057\u3044\u3067\u3059\u304b\uff1f\nUser: \u30d0\u30fc\u30ac\u30fc\u3068\u30e9\u30fc\u30b8\u30b5\u30a4\u30ba\u306e\u30d5\u30e9\u30a4\u30c9\u30dd\u30c6\u30c8\u3092\u304a\u9858\u3044\u3057\u307e\u3059\u3002\nSystem: \u627f\u77e5\u3044\u305f\u3057\u307e\u3057\u305f\u3002\u304a\u98f2\u307f\u7269\u306f\u4f55\u304b\u304a\u4ed8\u3051\u3057\u307e\u3059\u304b\uff1f\nUser: \u30e9\u30fc\u30b8\u30b5\u30a4\u30ba\u306e\u30b3\u30fc\u30e9\u3092\u304a\u9858\u3044\u3057\u307e\u3059\u3002\nSystem: \u5e97\u5185\u3067\u304a\u53ec\u3057\u4e0a\

## 評価

In [16]:
from llm_preference_extraction.evaluation import dialogue_evaluator, matching
from llm_preference_extraction.evaluation.dialogue_evaluator import collect_comparisons, compute_scores

dialogue_results = []

# 評価実行
for r in results:
    dialogue_id = r["dialogue_id"]
    ground_truths = r["ground_truth_annotation"]
    predictions = r["extracted_preferences"].get("preferences", [])

    # エンティティ&評価軸 マッチング
    matching_results = matching.find_optimal_matching(ground_truths, predictions)
    print(f"\n=== Dialogue {dialogue_id} ===")
    print(f"\nマッチング結果:")
    for gt_idx, pred_idx, pred, score in matching_results:
        print(f"  GT[{gt_idx}] -> Pred[{pred_idx}]  score={score:.2f}  matched={'Yes' if pred else 'No'}")

    # ペアごとの属性比較
    comparison = collect_comparisons(dialogue_id, ground_truths, predictions, matching_results)
    print(f"\n属性比較結果:")
    print(json.dumps(asdict(comparison), indent=2, default=str))

    # Precision, Recall, F1 計算
    result = compute_scores(comparison)
    print(f"\n評価計算結果")
    print(json.dumps(asdict(result), indent=2))
    dialogue_results.append(result)


=== Dialogue 3 ===

マッチング結果:
  GT[0] -> Pred[0]  score=10.00  matched=Yes
  GT[1] -> Pred[None]  score=0.00  matched=No
  GT[2] -> Pred[2]  score=10.00  matched=Yes

属性比較結果:
{
  "dialogue_id": 3,
  "n_gt": 3,
  "n_pred": 3,
  "comparisons": [
    {
      "is_matched": true,
      "is_axis_ok": false,
      "is_sub_axis_ok": false,
      "is_polarity_ok": false,
      "is_intensity_ok": false,
      "is_context_ok": true,
      "is_perfect_ok": false,
      "h_axis_gt": "{'wanting__goal', 'wanting'}",
      "h_axis_pred": "set()",
      "h_axis_partial_score": 0.0
    },
    {
      "is_matched": false,
      "is_axis_ok": false,
      "is_sub_axis_ok": false,
      "is_polarity_ok": false,
      "is_intensity_ok": false,
      "is_context_ok": false,
      "is_perfect_ok": false,
      "h_axis_gt": "{'wanting__goal', 'wanting'}",
      "h_axis_pred": "set()",
      "h_axis_partial_score": 0.0
    },
    {
      "is_matched": true,
      "is_axis_ok": false,
      "is_sub_axis_ok": fal